# SWANIRVAR v1.1.6 — Full-Stack Colab Launcher

**Purpose:** replace the old v1.1.2 frontend with the complete `swanirvar (1)` frontend,
preserve the v1.1.6 evidence-first backend, install the required backend for the new
frontend, validate the preserved ML/DL registry, and launch the complete portal.

**Important:** this notebook does not fabricate API keys, live data, datasets, or model
artifacts. Configure optional connectors before using their cloud features.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, sys, subprocess, socket, time, json, shutil

drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
print('Upload the generated Swanirvar_v1_1_6_FullStack ZIP')
uploaded = files.upload()
zip_files = list(Path('/content').glob('Swanirvar_v1_1_6_FullStack*.zip'))
if not zip_files:
    zip_files = list(Path('/content').glob('*.zip'))
assert zip_files, 'No ZIP uploaded.'
ZIP = zip_files[0]
print('Using:', ZIP)


In [ ]:
import zipfile, shutil
ROOT = Path('/content/swanirvar_v1_1_6_fullstack')
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
with zipfile.ZipFile(ZIP, 'r') as z:
    z.extractall(ROOT)
print('Extracted to:', ROOT)


In [ ]:
# If the ZIP contains a single project directory, use it as PROJECT.
children = [p for p in ROOT.iterdir()]
PROJECT = children[0] if len(children) == 1 and children[0].is_dir() else ROOT
print('PROJECT =', PROJECT)
for p in ['frontend', 'backend', 'server.ts', 'run_fullstack.py', 'requirements.txt']:
    q = PROJECT / p
    print(('OK ' if q.exists() else 'MISSING '), q)


In [ ]:
# Python dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT/'requirements.txt')], check=True)
print('Python dependencies installed.')


In [ ]:
# Node dependencies
subprocess.run(['npm', 'install', '--no-audit', '--no-fund', '--legacy-peer-deps'], cwd=str(PROJECT), check=True)
print('Node dependencies installed.')


## Optional API credentials

For Colab, the safest approach is to create Colab Secrets named:
- `GEMINI_API_KEY`
- `BHASHINI_INFERENCE_API_KEY`
- `BHASHINI_TOKEN`
- `DATA_GOV_IN_API_KEY`
- `AGMARKNET_API_KEY`
- `ENAM_API_KEY`
- `ORS_API_KEY`

The next cell copies any available Colab Secrets into the runtime environment.
Missing secrets are allowed.


In [ ]:
# Load available Colab Secrets without failing if a secret is absent.
try:
    from google.colab import userdata
    secret_names = [
        'GEMINI_API_KEY','BHASHINI_INFERENCE_API_KEY','BHASHINI_TOKEN',
        'BHASHINI_ASR_SERVICE_ID','BHASHINI_TTS_SERVICE_ID',
        'BHASHINI_ASR_SERVICE_ID_BN','BHASHINI_ASR_SERVICE_ID_HI',
        'BHASHINI_ASR_SERVICE_ID_TA','BHASHINI_ASR_SERVICE_ID_TE',
        'BHASHINI_TTS_SERVICE_ID_BN','BHASHINI_TTS_SERVICE_ID_HI',
        'BHASHINI_TTS_SERVICE_ID_TA','BHASHINI_TTS_SERVICE_ID_TE',
        'DATA_GOV_IN_API_KEY','AGMARKNET_API_KEY','ENAM_API_KEY','ORS_API_KEY'
    ]
    loaded = []
    for name in secret_names:
        try:
            value = userdata.get(name)
            if value:
                os.environ[name] = value
                loaded.append(name)
        except Exception:
            pass
    print('Loaded secrets:', loaded if loaded else 'none')
except Exception as e:
    print('Colab Secrets unavailable; continuing without cloud credentials.')


In [ ]:
# Validate preserved ML/DL registry and evidence files.
registry = PROJECT / 'backend' / 'data' / 'ml_dl' / 'registry.json'
assert registry.exists(), 'ML registry missing.'
reg = json.loads(registry.read_text(encoding='utf-8'))
print(json.dumps(reg, indent=2))

print('\nML registry status:')
for model in reg.get('models', []):
    print('-', model.get('id'), '|', model.get('type'), '|', model.get('status'))

evidence = PROJECT / 'backend' / 'data' / 'evidence'
print('\nEvidence files:')
for p in sorted(evidence.glob('*')):
    print('-', p.name)


## Optional ML training hook

The preserved v1.1.6 registry intentionally does **not** fabricate binary model files.
If the real full Census training dataset exists in your preserved Drive workspace, this
cell only reports its presence. Actual training should be run from the preserved ML
workflow against that real dataset; otherwise this build stays in inference-from-
preserved-real-outputs mode.


In [ ]:
drive_ml = Path('/content/drive/MyDrive/Swanirvar')
candidates = list(drive_ml.rglob('*')) if drive_ml.exists() else []
data_candidates = [p for p in candidates if p.is_file() and p.suffix.lower() in {'.csv','.parquet','.xlsx'} and 'census' in p.name.lower()]
print('Census-like datasets found:', len(data_candidates))
for p in data_candidates[:20]:
    print(' -', p)

if not data_candidates:
    print('No full Census training dataset found. No model will be fabricated or trained.')


In [ ]:
# Start the complete stack.
runtime = PROJECT / 'runtime'
runtime.mkdir(exist_ok=True)

# Stop stale listeners when possible.
subprocess.run(['bash','-lc','fuser -k 8000/tcp 2>/dev/null || true; fuser -k 3000/tcp 2>/dev/null || true'], check=False)

env = os.environ.copy()
env['SWANIRVAR_BACKEND_URL'] = 'http://127.0.0.1:8000'
env['SWANIRVAR_UI_PORT'] = '3000'

log_path = runtime / 'colab_launcher.log'
log = open(log_path, 'w', encoding='utf-8')

proc = subprocess.Popen(
    [sys.executable, str(PROJECT/'run_fullstack.py')],
    cwd=str(PROJECT),
    env=env,
    stdout=log,
    stderr=subprocess.STDOUT
)

def wait_port(port, timeout=40):
    end = time.time() + timeout
    while time.time() < end:
        try:
            with socket.create_connection(('127.0.0.1', port), timeout=1):
                return True
        except OSError:
            time.sleep(.5)
    return False

assert wait_port(8000), 'FastAPI backend did not start.'
assert wait_port(3000), 'Frontend server did not start.'
print('Backend and frontend are running.')


In [ ]:
from google.colab import output
backend_url = output.eval_js("google.colab.kernel.proxyPort(8000)")
frontend_url = output.eval_js("google.colab.kernel.proxyPort(3000)")

print('BACKEND PROXY:')
print(backend_url)
print('\nFRONTEND PROXY:')
print(frontend_url)

# Quick health checks through localhost before opening the browser.
import requests
print('\nBackend health:', requests.get('http://127.0.0.1:8000/health', timeout=10).json())
print('Frontend health:', requests.get('http://127.0.0.1:3000/api/health', timeout=10).json())
print('Full-stack bridge:', requests.get('http://127.0.0.1:3000/api/fullstack/health', timeout=10).json())


In [ ]:
# Open the frontend portal.
output.eval_js(f"window.open({frontend_url!r}, '_blank')")
print('Opened Swanirvar frontend.')


## Runtime notes

- Keep this Colab runtime active while using the portal.
- The Colab frontend URL changes after a new runtime.
- API connectors that require credentials remain unavailable until configured.
- The v1.1.6 backend remains evidence-first and deterministic; ML/DL is advisory.
- For setup/API instructions see `docs/SETUP.md`, `docs/API_INTEGRATION.md`, and
  `docs/ML_GUIDE.md` inside the ZIP.
